# Paper Figure 6 -- motif reproducibility, support and robustness

Are the interpreted motifs M0-M4 reproducible, broadly supported and robust to condition composition?

Upstream analyses:
- `analysis/08_multiseed_stability/k46_stability/k46_cross_seed_stability.py` (cross-seed alignment)
- `revision_06_archetype_interpretation_K46_independent_groups` (observation-level support)
- `revision_09_condition_influence_and_seed_stability_K46` (leave-one-condition-out refits)

This notebook is presentation-only: it loads those outputs and assembles the figure.

Panels:
- **A:** cross-seed motif recovery -- spatial-map Pearson r of each motif with its Hungarian-matched archetype in
  each further fit seed (the full three-metric version is supplementary figure S04)
- **B:** broad observation-level support -- spatial support of each motif in each of the 108 participant-condition
  observations (36 participants x 3 conditions)
- **C:** condition-specific deployment -- mean support per condition (+/- SEM over the 36 observations)
- **D:** leave-one-condition-out similarity -- spatial-map Pearson r of each motif after refitting without one
  condition (same metric as supplementary figure S08)

The edge-trimming panels that used to be D and E are now in the edge-trim supplement
(`paper/supplement/supp_05_edge_trim_diagnostics.ipynb`).

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sys
sys.path.insert(0, "src")
from msaa_pipeline.paper_archetypes import paper_archetypes
# M0-M4, in the K=46 fit's own numbering (canonical 28,35,16,39,45; legacy 4,5,7,8,9)
ARCHETYPES, ARCHETYPE_LABELS = paper_archetypes()
LABEL = dict(zip(ARCHETYPES, ARCHETYPE_LABELS))
CONDITIONS = ["intact", "word", "rest"]
COND_LABEL = {"intact": "Intact narrative", "word": "Word-scrambled", "rest": "Rest"}
COLORS = {"intact": "#7B3294", "word": "#2CA25F", "rest": "#222222"}   # paper condition colours
RECOVERED_R = 0.9

SUPPORT_DIR = Path("revision_05_archetype_interpretation_and_support")
CI_DIR = Path("outputs/interpretation/condition_influence_and_seed_stability_K46")
# cross-seed alignment of M0-M4 (k46_cross_seed_stability.py); override with MSAA_SEED_ALIGNMENT
SEED_ALIGN_CSV = Path(os.environ.get(
    "MSAA_SEED_ALIGNMENT",
    "outputs/multiseed_stability/k46_cross_seed_stability/K46_paper_archetypes_alignment.csv"))

SUPPORT_CSV = SUPPORT_DIR / "selected_archetype_support_by_independent_participant.csv"
COND_CSV = SUPPORT_DIR / "selected_archetype_condition_summary_independent.csv"
LOCO_CSV = CI_DIR / "leave_one_condition_out_archetype_comparison.csv"

OUTPUT_DIR = Path("outputs/paper_figures")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_PNG = OUTPUT_DIR / "Figure6_selected_motif_support_robustness.png"
FIG_PDF = OUTPUT_DIR / "Figure6_selected_motif_support_robustness.pdf"

for p in [SUPPORT_CSV, COND_CSV, LOCO_CSV, SEED_ALIGN_CSV]:
    if not p.exists():
        raise FileNotFoundError(
            f"Missing required input: {p}\n"
            "Run the corresponding upstream analysis first."
        )

support = pd.read_csv(SUPPORT_CSV)
cond_summary = pd.read_csv(COND_CSV)
loco = pd.read_csv(LOCO_CSV)
align = pd.read_csv(SEED_ALIGN_CSV)

In [ ]:
# Derived tables for plotting
support = support[support["archetype"].isin(ARCHETYPES)].copy()
assert support.groupby("archetype").size().eq(108).all()      # 36 participants x 3 conditions per motif

align = align[align["seed"] > 0]
SEEDS = sorted(align["seed"].unique())
seed_r = align.pivot(index="paper_label", columns="seed", values="map_r").loc[ARCHETYPE_LABELS, SEEDS]

# leave-one-condition-out: spatial-map Pearson r with the full-data motif, Hungarian-matched on the generators
# and averaged over the two conditions present in both fits (revision_09); the same values as S08
loco = loco[loco["reference_archetype"].isin(ARCHETYPES)]
loco_r = (loco.assign(dropped=loco["comparison"].str.replace("drop_", "", regex=False))
          .pivot(index="reference_archetype", columns="dropped", values="mean_spatial_map_correlation")
          .reindex(index=ARCHETYPES, columns=CONDITIONS))

support.to_csv(OUTPUT_DIR / "Figure6_support_source.csv", index=False)
seed_r.to_csv(OUTPUT_DIR / "Figure6_cross_seed_map_r_source.csv")
loco_r.rename(index=LABEL).to_csv(OUTPUT_DIR / "Figure6_loco_map_r_source.csv")
display(seed_r.round(3)); display(loco_r.rename(index=LABEL).round(3))

In [ ]:
import matplotlib.colorbar as _mcb
_mcb.Colorbar.n_rasterize = 10**9  # keep colorbars vector in PDFs
_add_solids = _mcb.Colorbar._add_solids


def _add_solids_no_seams(self, *a, **k):
    # stroke each colour band in its own colour: no hairline gaps in PDF viewers (matplotlib colorbar docs)
    _add_solids(self, *a, **k)
    if getattr(self, "solids", None) is not None:
        self.solids.set_edgecolor("face")
        self.solids.set_linewidth(0.3)


_mcb.Colorbar._add_solids = _add_solids_no_seams


def vheat(ax, arr, origin="upper", **kw):
    """imshow look-alike drawn as vector cells (pcolormesh), so the PDF holds no raster image."""
    a = np.ma.masked_invalid(np.asarray(arr, dtype=float))
    n, m = a.shape
    im = ax.pcolormesh(np.arange(m + 1) - 0.5, np.arange(n + 1) - 0.5, a, shading="flat",
                       edgecolors="face", linewidth=0.3, **kw)  # face-coloured edges: no seams
    ax.set_xlim(-0.5, m - 0.5)
    ax.set_ylim((-0.5, n - 0.5) if origin == "lower" else (n - 0.5, -0.5))
    return im

# Drawn at the journal's print width (Network Neuroscience reproduces figures ~5 in across),
# so the point sizes below are the printed sizes.
FS_SUPTITLE, FS_TITLE, FS_LABEL, FS_TICK, FS_ANNOT, FS_LETTER = 9, 8, 7.5, 7, 6, 10
plt.rcParams.update({"axes.linewidth": 0.6, "xtick.major.width": 0.6, "ytick.major.width": 0.6,
                     "xtick.major.size": 2.5, "ytick.major.size": 2.5,
                     "xtick.major.pad": 1.5, "ytick.major.pad": 1.5})
fig = plt.figure(figsize=(4.25, 4.6), constrained_layout=True)  # final width after the tight crop is ~5 in
fig.get_layout_engine().set(w_pad=0.03, h_pad=0.04, wspace=0.06, hspace=0.06)
gs = fig.add_gridspec(2, 2, width_ratios=[1.4, 1])
axA = fig.add_subplot(gs[0, 0])
axB = fig.add_subplot(gs[0, 1])
axC = fig.add_subplot(gs[1, 0])
axD = fig.add_subplot(gs[1, 1])
x = np.arange(len(ARCHETYPES))
offsets = {"intact": -0.24, "word": 0.0, "rest": 0.24}


def style(ax):
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", color="0.9", lw=0.5)
    ax.set_axisbelow(True)


# A - cross-seed motif recovery (spatial-map r with the seed-0 motif)
g = seed_r.to_numpy(dtype=float)
im = vheat(axA, np.clip(g, 0, 1), cmap="Purples", vmin=0, vmax=1)
for i in range(g.shape[0]):
    for j in range(g.shape[1]):
        v = g[i, j]
        axA.text(j, i, f"{v:.2f}".replace("0.", ".").replace("1.00", "1"), ha="center", va="center",
                 fontsize=FS_ANNOT, color="white" if v > 0.6 else "#222222")
        if v < RECOVERED_R:
            axA.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False, ec="#e34a33", lw=1.2))
axA.set_xticks(range(len(SEEDS)))
axA.set_xticklabels(SEEDS)
axA.set_yticks(range(len(ARCHETYPE_LABELS)))
axA.set_yticklabels(ARCHETYPE_LABELS)
axA.tick_params(length=0)
axA.set_xlabel("Fit seed", fontsize=FS_LABEL, labelpad=1)
axA.set_title("Cross-seed motif recovery", fontweight="bold", loc="left", fontsize=FS_TITLE)
cb = fig.colorbar(im, ax=axA, shrink=0.95, aspect=18, pad=0.015, fraction=0.06)
cb.set_label("Spatial-map r\nwith seed-0 motif", fontsize=FS_LABEL, labelpad=2)
cb.set_ticks([0, 0.5, 1])
cb.ax.tick_params(labelsize=FS_TICK)
cb.outline.set_linewidth(0.6)

# B - observation-level support (one point per participant-condition observation)
rng = np.random.default_rng(42)
box_data = []
for pos, k in enumerate(ARCHETYPES):
    cur = support[support["archetype"] == k]
    box_data.append(cur["support_l2"].to_numpy(dtype=float))
    for condition in CONDITIONS:
        vals = cur.loc[cur["condition"] == condition, "support_l2"].to_numpy(dtype=float)
        axB.scatter(np.full(len(vals), pos) + rng.normal(0, 0.06, size=len(vals)), vals, s=3, alpha=0.5,
                    color=COLORS[condition], lw=0)
axB.boxplot(box_data, positions=x, widths=0.5, showfliers=False,
            medianprops=dict(color="#222222", lw=1.0), boxprops=dict(color="0.35", lw=0.7),
            whiskerprops=dict(color="0.35", lw=0.7), capprops=dict(color="0.35", lw=0.7))
axB.set_xticks(x)
axB.set_xticklabels(ARCHETYPE_LABELS)
axB.set_ylabel(r"Spatial support $\|W^{(s)}[k,:]\|_2$", fontsize=FS_LABEL, labelpad=1)
axB.set_title("Broad observation-level support", fontweight="bold", loc="left", fontsize=FS_TITLE - 0.5)
style(axB)

# C - condition-specific deployment (mean +- SEM over the 36 observations per condition)
for condition in CONDITIONS:
    cur = cond_summary[cond_summary["condition"] == condition].set_index("archetype").reindex(ARCHETYPES)
    axC.errorbar(x + offsets[condition], cur["mean_support"], yerr=cur["sem_support"], marker="o", ms=3.2,
                 linestyle="none", capsize=1.5, elinewidth=0.8, capthick=0.8, color=COLORS[condition],
                 label=COND_LABEL[condition])
axC.set_xticks(x)
axC.set_xticklabels(ARCHETYPE_LABELS)
axC.set_ylabel(r"Mean spatial support $\|W^{(s)}[k,:]\|_2$", fontsize=FS_LABEL, labelpad=1)
axC.set_title("Condition-specific deployment", fontweight="bold", loc="left", fontsize=FS_TITLE)
style(axC)

# D - leave-one-condition-out similarity
for k_, condition in enumerate(CONDITIONS):
    axD.bar(x + offsets[condition], loco_r[condition], width=0.23, color=COLORS[condition])
axD.axhline(RECOVERED_R, color="#E69F00", ls="--", lw=1.0)   # r = 0.90 recovery threshold (amber: red means not recovered in A)
axD.set_xticks(x)
axD.set_xticklabels(ARCHETYPE_LABELS)
axD.set_ylim(0, 1.02)
axD.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
axD.set_ylabel("Spatial-map r with\nfull-data motif", fontsize=FS_LABEL, labelpad=1)
axD.set_title("Leave-one-condition-out stability", fontweight="bold", loc="left", fontsize=FS_TITLE - 0.5)
style(axD)

h = [plt.Rectangle((0, 0), 1, 1, color=COLORS[c]) for c in CONDITIONS] + [
    plt.Line2D([], [], color="#E69F00", ls="--", lw=1.0)]
fig.legend(h, [COND_LABEL[c] for c in CONDITIONS] + ["Recovery threshold (r = 0.9)"], frameon=False, ncol=4, loc="upper center",
           bbox_to_anchor=(0.5, 0.0), fontsize=FS_LABEL, handlelength=1.0, handleheight=0.8,
           handletextpad=0.4, columnspacing=0.9,
           title="Condition (B, C) / condition left out (D)", title_fontsize=FS_TICK)

for ax, letter in zip([axA, axB, axC, axD], "ABCD"):
    ax.text(-0.16, 1.04, letter, transform=ax.transAxes, fontsize=FS_LETTER, fontweight="bold", va="bottom")
    ax.tick_params(labelsize=FS_TICK)
axA.tick_params(axis="y", labelsize=FS_LABEL)

fig.suptitle("Support and robustness of motifs M0\u2013M4", fontsize=FS_SUPTITLE, fontweight="bold")
fig.savefig(FIG_PNG, dpi=600, bbox_inches="tight", pad_inches=0.02)
fig.savefig(FIG_PDF, dpi=600, bbox_inches="tight", pad_inches=0.02)
plt.show()

print("Saved:", FIG_PNG)
print("Saved:", FIG_PDF)


### Figure 6 interpretation

Panel A asks whether the motifs are reproducible across initializations (consistency across fit seeds, not
independent samples). Panel B asks whether a small number of observations drives them; the unit is the
participant-condition observation, not a unique participant. Panels C and D show how the motifs are
deployed in each condition and whether they survive refitting without one condition.

Report the motifs as they come out; do not swap motifs post hoc. Edge-trim sensitivity is in the supplement.